# nanolab — train Qwen3-0.6B on gsm8k (GRPO + LoRA)

Works on **Colab** or **Kaggle**.

**Before running:**
- Colab: *Runtime → Change runtime type → T4 GPU → Save*
- Kaggle: *Settings → Accelerator → GPU T4* and turn **Internet ON** (needs phone-verified account)

Run cells **one by one, top to bottom** (don't use Run-all — it would fire the
resume cell too). If the session disconnects mid-training: reconnect, re-run
cells 1–3, then run the **resume** cell (4b) instead of the train cell (4).
With the Drive cell (1b) enabled, checkpoints survive any disconnect.

In [ ]:
# 1) Get the code (repo is public — no token needed)
!rm -rf nanolab && git clone https://github.com/khwahish1509/RLPost.git nanolab
%cd nanolab

In [ ]:
# 1b) Colab only, strongly recommended: keep checkpoints + the db on Google
#     Drive so a dead VM loses NOTHING (resume just works).
#     Approve the popup when it asks for Drive access. Skipped on Kaggle.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    base = '/content/drive/MyDrive/nanolab'
    os.makedirs(f'{base}/adapters', exist_ok=True)
    os.makedirs(f'{base}/results', exist_ok=True)
    !rm -rf adapters results && ln -s {base}/adapters adapters && ln -s {base}/results results
    print('checkpoints and db now persist in Drive: MyDrive/nanolab/')
except ImportError:
    print('not Colab — skipping Drive mount (Kaggle keeps /kaggle/working per session)')

In [ ]:
# 2) Install: uv + project deps + the GPU training stack.
#    `uv add` (not `uv pip install`) so the GPU stack becomes part of the
#    project on this machine and can't be stripped by a later sync.
import os
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv sync -q
!uv add torch transformers peft
!uv tool install prime
# verify NOW, not in cell 4 — must print 'CUDA available: True'
!uv run python -c "import torch; print('torch OK — CUDA available:', torch.cuda.is_available())"

In [ ]:
# 3) Install the task environment + quick sanity check
!uv run nanolab env install primeintellect/gsm8k
!uv run pytest -q  # 29 tests incl. a GRPO direction check

In [ ]:
# 4) TRAIN — 50 steps. Watch for:
#    'pre-flight baseline reward: 0.xxx'  (must be in the 10–80% window)
#    'step N reward 0.xxx loss y.yyyy'    (one line per step)
#    checkpoints land in adapters/ every 10 steps
!uv run nanolab train configs/qwen3-0.6b-gsm8k.toml

In [ ]:
# 4b) ONLY after a disconnect: continue where it stopped
#     (re-run cells 1, 1b, 2, 3 first; same batches get redrawn)
!uv run nanolab train configs/qwen3-0.6b-gsm8k.toml --resume

In [ ]:
# 5) Bring the results home: a zip with the trained adapter checkpoints
#    + the lab database. Unzip it into your local nanolab folder.
#    (With Drive enabled, everything is ALSO in MyDrive/nanolab/ already.)
!zip -qr ../nanolab-artifacts.zip adapters/ results/nanolab.db
try:
    from google.colab import files  # Colab: triggers a browser download
    files.download('../nanolab-artifacts.zip')
except ImportError:                 # Kaggle: grab it from the Output panel
    print('Kaggle: nanolab-artifacts.zip is one directory up —')
    print('download it from the notebook Output panel.')